In [7]:
import pandas as pd

df = pd.read_excel("../data/raw/Analytics Case Study Dataset.xlsx")

df.head()

,Agmt Id,Cust Age,Cust Gender,Cust Cibil Score,Cust Employment Type,Cust Net Salary,Coborrower Flag,App Score Risk,Agmt Date,Seizure Date,...,OS Balance At Liquidation,Asset Age Months At Seizure,Months Spent In Yard,Asset Bodycondition,Asset Tyrecondition,Asset Generalcondition,Asset Enginecondition,Asset Accident Flag,Traiffic Challan Amount,Target Sold Amount At Liquidation
0,ASSET_1,29,F,-1,NREGI,30000,N,LOW RISK,2023-12-27,2025-05-23,...,46934.0,17.100000,1.033333,G,G,G,G,No,0,37000.0
1,ASSET_2,24,M,-1,NREGI,60000,N,LOW RISK,2025-11-20,2026-05-31,...,85177.0,6.400000,1.966667,G,G,G,G,No,0,56000.0
2,ASSET_3,32,F,763,AGR,29500,N,LOW RISK,2024-05-16,2025-06-24,...,51678.1,13.466667,2.066667,A,A,A,A,No,0,40500.0
3,ASSET_4,25,F,-1,NREGI,40000,N,MEDIUM RISK,2025-10-24,2026-04-30,...,65874.0,6.266667,1.900000,A,A,A,A,No,0,49000.0
4,ASSET_5,59,F,653,SAL,45000,N,MEDIUM RISK,2024-09-20,2025-05-30,...,66571.0,8.400000,1.833333,G,G,G,G,No,0,46000.0


Asset Health Index

In [8]:
condition_map = {
    "G": 100,
    "A": 70,
    "P": 30
}

df["Asset_Health_Index"] = (
      0.35 * df["Asset Enginecondition"].map(condition_map)
    + 0.25 * df["Asset Generalcondition"].map(condition_map)
    + 0.20 * df["Asset Bodycondition"].map(condition_map)
    + 0.20 * df["Asset Tyrecondition"].map(condition_map)
)

df["Asset_Health_Index"].describe()

count    15000.000000
mean        83.446667
std         17.802456
min         30.000000
25%         70.000000
50%         86.500000
75%        100.000000
max        100.000000
Name: Asset_Health_Index, dtype: float64

Recovery Efficiency Index

In [9]:
df["Recovery_Efficiency_Index"] = (
    df["Target Sold Amount At Liquidation"]
    /
    df["OS Balance At Liquidation"]
) * 100

df["Recovery_Efficiency_Index"].describe()

count    15000.000000
mean        70.075240
std         50.836102
min          8.440956
25%         48.114216
50%         59.271900
75%         75.695404
max       1171.303075
Name: Recovery_Efficiency_Index, dtype: float64

Loss Given Default (LGD)

In [10]:

df["LGD"] = (
    df["OS Balance At Liquidation"]
    -
    df["Target Sold Amount At Liquidation"]
)

df["LGD"].describe()

count     15000.000000
mean      31041.681920
std       25736.933179
min      -73972.000000
25%       13651.007500
50%       30648.725000
75%       47574.675000
max      224180.000000
Name: LGD, dtype: float64

LGD Percentage

In [11]:
df["LGD_pct"] = (
    (
        df["OS Balance At Liquidation"]
        -
        df["Target Sold Amount At Liquidation"]
    )
    /
    df["OS Balance At Liquidation"]
) * 100

df["LGD_pct"].describe()

count    15000.000000
mean        29.924760
std         50.836102
min      -1071.303075
25%         24.304596
50%         40.728100
75%         51.885784
max         91.559044
Name: LGD_pct, dtype: float64

Asset Depreciation %

In [12]:
df["Depreciation_Pct"] = (
    (
        df["Asset Cost At Disbursal"]
        -
        df["Target Sold Amount At Liquidation"]
    )
    /
    df["Asset Cost At Disbursal"]
) * 100

df["Depreciation_Pct"].describe()

count    15000.000000
mean        60.720310
std         10.763882
min        -27.958298
25%         53.938999
50%         60.880393
75%         67.827090
max         93.602047
Name: Depreciation_Pct, dtype: float64

Asset Age Group

In [13]:
df["Asset_Age_Group"] = pd.cut(
    df["Asset Age Months At Seizure"],
    bins=[0,12,24,36,60,120],
    labels=[
        "0-1 Year",
        "1-2 Years",
        "2-3 Years",
        "3-5 Years",
        "5+ Years"
    ]
)

df["Asset_Age_Group"].value_counts()

Asset_Age_Group
1-2 Years    6267
0-1 Year     4691
2-3 Years    3001
3-5 Years    1010
5+ Years       31
Name: count, dtype: int64

Accident Score

In [14]:

df["Accident_Score"] = np.where(
    df["Asset Accident Flag"]=="Yes",
    100,
    0
)

df["Accident_Score"].value_counts()


NameError: name 'np' is not defined

Residual Risk Score

In [ ]:

from sklearn.preprocessing import MinMaxScaler

risk_features = pd.DataFrame({
    "age": df["Asset Age Months At Seizure"],
    "lgd": df["LGD_pct"],
    "challan": df["Traiffic Challan Amount"],
    "health": 100 - df["Asset_Health_Index"]
})

scaler = MinMaxScaler()

scaled = scaler.fit_transform(risk_features)

df["Residual_Risk_Score"] = (
      scaled[:,0] * 0.35
    + scaled[:,1] * 0.35
    + scaled[:,2] * 0.10
    + scaled[:,3] * 0.20
) * 100

df["Residual_Risk_Score"].describe()


Risk Band

In [ ]:

def risk_band(score):

    if score <= 25:
        return "Low"

    elif score <= 50:
        return "Medium"

    elif score <= 75:
        return "High"

    else:
        return "Critical"


df["Risk_Band"] = df["Residual_Risk_Score"].apply(risk_band)

df["Risk_Band"].value_counts()


 Profitability Score



In [ ]:

profitability_raw = (
    df["Recovery_Efficiency_Index"]
    -
    df["LGD_pct"]
)

profitability_raw = profitability_raw.fillna(
    profitability_raw.median()
)

from sklearn.preprocessing import MinMaxScaler

profit_scaler = MinMaxScaler()

df["Profitability_Score"] = (
    profit_scaler.fit_transform(
        profitability_raw.values.reshape(-1,1)
    ) * 100
)


Quick Validation


In [ ]:
engineered_cols = [

    "Asset_Health_Index",

    "Recovery_Efficiency_Index",

    "LGD",

    "LGD_pct",

    "Depreciation_Pct",

    "Residual_Risk_Score",

    "Risk_Band",

    "Profitability_Score"

]

df[engineered_cols].head()

In [18]:
df.to_csv(
    "../data/processed/final_dataset.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully
